# Step 14 — final submission validation

This notebook checks the final repository structure and provides an executable Airflow validation sequence. It does not edit source data or the read-only snapshots.

Run the Astro command cell only when Docker Desktop and the Astro CLI are installed.

In [1]:
# Resolve the repository root so the notebook works from VS Code or Jupyter.
from pathlib import Path
import ast
import re
import shutil
import subprocess

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'SUBMISSION.md').exists():
    REPO_ROOT = Path.cwd().parent
print(f'Repository: {REPO_ROOT}')

Repository: /Users/kumarsamanvaya/Documents/IITR - FDE Course/Data Pipepline Assignment/_Samy_IIT_Roorkie_CEC_Futurenece_Data_Pipeline_to_ML_Model


In [2]:
# Verify every artifact required by the assignment is present.
required_paths = [
    REPO_ROOT / 'dags' / 'nammamart_etl_samy.py',
    REPO_ROOT / 'dags' / 'nammamart_elt_samy.py',
    REPO_ROOT / 'include' / 'sql',
    REPO_ROOT / 'include' / 'data' / 'nammamart',
    REPO_ROOT / 'include' / 'quarantine' / 'samy',
    REPO_ROOT / 'include' / 'warehouse_samy.duckdb',
    REPO_ROOT / 'SUBMISSION.md',
]
missing_paths = [str(path.relative_to(REPO_ROOT)) for path in required_paths if not path.exists()]
assert not missing_paths, f'Missing final artifacts: {missing_paths}'
print(f'Final artifact check passed: {len(required_paths)} required paths present.')

Final artifact check passed: 7 required paths present.


In [3]:
# Parse DAG source without importing Airflow. This catches Python syntax
# errors even on a machine that has not started the Airflow containers.
dag_files = {
    'nammamart_etl_samy': REPO_ROOT / 'dags' / 'nammamart_etl_samy.py',
    'nammamart_elt_samy': REPO_ROOT / 'dags' / 'nammamart_elt_samy.py',
}
for dag_id, dag_path in dag_files.items():
    ast.parse(dag_path.read_text(), filename=str(dag_path))
    source = dag_path.read_text()
    assert f'dag_id="{dag_id}"' in source, f'DAG ID not found in {dag_path}'
    assert 'schedule="0 9 * * 1-6"' in source, f'Schedule missing in {dag_path}'
    print(f'{dag_path.name}: syntax, DAG ID and schedule checks passed')

nammamart_etl_samy.py: syntax, DAG ID and schedule checks passed
nammamart_elt_samy.py: syntax, DAG ID and schedule checks passed


In [4]:
# Display the exact runtime commands required by Step 14.
commands = [
    ['astro', 'dev', 'start'],
    ['astro', 'dev', 'run', 'dags', 'list-import-errors'],
    ['astro', 'dev', 'run', 'dags', 'test', 'nammamart_etl_samy'],
    ['astro', 'dev', 'run', 'dags', 'test', 'nammamart_elt_samy'],
]
for command in commands:
    print(' '.join(command))

astro dev start
astro dev run dags list-import-errors
astro dev run dags test nammamart_etl_samy
astro dev run dags test nammamart_elt_samy


In [5]:
# Run the Astro sequence when the CLI is available. The guard keeps the
# structural notebook test usable on machines without Astro installed.
astro = shutil.which('astro')
if astro is None:
    print('Astro CLI not found; install Astro CLI and Docker Desktop, then rerun this cell.')
else:
    for command in commands:
        print(f'\n$ {" ".join(command)}')
        completed = subprocess.run(command, cwd=REPO_ROOT, text=True, capture_output=True)
        print(completed.stdout)
        print(completed.stderr)
        if completed.returncode != 0:
            raise RuntimeError(f'Command failed with exit code {completed.returncode}: {command}')
    print('Astro validation passed: no import errors and both DAG tests completed.')


$ astro dev start


➤ Airflow UI: http://samy-iit-roorkie-cec-futurenece-data-pipeline-to-ml-model.localhost:6563
➤ Postgres Database: postgresql://localhost:5432/postgres
➤ The default Postgres DB credentials are: postgres:postgres



$ astro dev run dags list-import-errors


Running: airflow dags list-import-errors
2026-10-22T13:53:13.103657Z [warning  ] Astro managed secrets backend is disabled [astronomer.runtime.plugin.AstronomerRuntimePlugin] loc=plugin.py:125
2026-10-22T13:53:13.134716Z [warning  ] FileType is deprecated. Simply open files after parsing arguments. [py.warnings] category=PendingDeprecationWarning filename=/usr/local/lib/python3.14/site-packages/airflow/cli/cli_config.py lineno=646
2026-10-22T13:53:13.136299Z [warning  ] FileType is deprecated. Simply open files after parsing arguments. [py.warnings] category=PendingDeprecationWarning filename=/usr/local/lib/python3.14/site-packages/airflow/cli/cli_config.py lineno=868
2026-10-22T13:53:13.676760Z [warning  ] Could not import graphviz. Rendering graph to the graphical format will not be possible. 
You might need to install the graphviz package and necessary system packages.
Run `pip install graphviz` to attempt to install it. [py.warnings] category=UserWarning filename=/usr/local/lib/pyt

Running: airflow dags test nammamart_etl_samy
2026-10-22T13:53:17.003799Z [warning  ] Astro managed secrets backend is disabled [astronomer.runtime.plugin.AstronomerRuntimePlugin] loc=plugin.py:125
2026-10-22T13:53:17.022560Z [warning  ] FileType is deprecated. Simply open files after parsing arguments. [py.warnings] category=PendingDeprecationWarning filename=/usr/local/lib/python3.14/site-packages/airflow/cli/cli_config.py lineno=646
2026-10-22T13:53:17.023062Z [warning  ] FileType is deprecated. Simply open files after parsing arguments. [py.warnings] category=PendingDeprecationWarning filename=/usr/local/lib/python3.14/site-packages/airflow/cli/cli_config.py lineno=868
2026-10-22T13:53:17.420842Z [warning  ] Could not import graphviz. Rendering graph to the graphical format will not be possible. 
You might need to install the graphviz package and necessary system packages.
Run `pip install graphviz` to attempt to install it. [py.warnings] category=UserWarning filename=/usr/local/li

Running: airflow dags test nammamart_elt_samy
2026-10-22T13:53:32.785759Z [warning  ] Astro managed secrets backend is disabled [astronomer.runtime.plugin.AstronomerRuntimePlugin] loc=plugin.py:125
2026-10-22T13:53:32.797197Z [warning  ] FileType is deprecated. Simply open files after parsing arguments. [py.warnings] category=PendingDeprecationWarning filename=/usr/local/lib/python3.14/site-packages/airflow/cli/cli_config.py lineno=646
2026-10-22T13:53:32.797758Z [warning  ] FileType is deprecated. Simply open files after parsing arguments. [py.warnings] category=PendingDeprecationWarning filename=/usr/local/lib/python3.14/site-packages/airflow/cli/cli_config.py lineno=868
2026-10-22T13:53:33.099797Z [warning  ] Could not import graphviz. Rendering graph to the graphical format will not be possible. 
You might need to install the graphviz package and necessary system packages.
Run `pip install graphviz` to attempt to install it. [py.warnings] category=UserWarning filename=/usr/local/li